In [1]:
!pip install facenet-pytorch

In [2]:
from facenet_pytorch import MTCNN, InceptionResnetV1

print("FaceNet OK")

[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


FaceNet OK


In [3]:
from google.colab import files

uploaded = files.upload()

Saving Ravi.jpg to Ravi (2).jpg
Saving Priya.jpg to Priya (2).jpg
Saving Suresh.jpg to Suresh (2).jpg
Saving Arjun.jpg to Arjun (2).jpg
Saving Lokes.jpg to Lokes (2).jpg
Saving Ragunath.jpg to Ragunath (3).jpg
Saving Lakshmi.jpg to Lakshmi (2).jpg


In [4]:
from PIL import Image

mtcnn = MTCNN(image_size=160)

resnet = InceptionResnetV1(
    pretrained='vggface2'
).eval()

print("Models Loaded")

Models Loaded


Face Embedding Function

In [5]:
def get_embedding(image_path):

    img = Image.open(image_path)

    face = mtcnn(img)

    if face is None:
        return None

    face = face.unsqueeze(0)

    embedding = resnet(face)

    return embedding.detach().numpy()

Create Database

In [6]:
resident_db = {}

Register Residents

In [7]:
resident_db["Ravi"] = get_embedding("Ravi.jpg")
resident_db["Priya"] = get_embedding("Priya.jpg")
resident_db["Suresh"] = get_embedding("Suresh.jpg")
resident_db["Arjun"] = get_embedding("Arjun.jpg")
resident_db["Lokes"] = get_embedding("Lokes.jpg")
resident_db["Ragunath"] = get_embedding("Ragunath.jpg")
resident_db["Lakshmi"] = get_embedding("Lakshmi.jpg")

print("Residents Registered Successfully")

Residents Registered Successfully


Verify Database

In [8]:
print(resident_db.keys())

dict_keys(['Ravi', 'Priya', 'Suresh', 'Arjun', 'Lokes', 'Ragunath', 'Lakshmi'])


Check Number of Residents

In [9]:
print("Total Residents:", len(resident_db))

Total Residents: 7


Visitor Verification

In [10]:
from numpy.linalg import norm

def verify_visitor(visitor_image):

    visitor_embedding = get_embedding(visitor_image)

    if visitor_embedding is None:
        return "No Face Detected"

    best_match = None
    best_score = 999

    for resident_name, resident_embedding in resident_db.items():

        distance = norm(
            visitor_embedding - resident_embedding
        )

        if distance < best_score:
            best_score = distance
            best_match = resident_name

    if best_score < 1.0:
        return f"AUTHORIZED: {best_match} | Score: {best_score:.2f}"

    else:
        return f"UNAUTHORIZED | Score: {best_score:.2f}"

Apartment Mapping

In [11]:
resident_apartment = {
    "Ravi": "A101",
    "Lakshmi": "A101",
    "Priya": "A102",
    "Suresh": "A102",
    "Arjun": "A103",
    "Lokes": "A104",
    "Ragunath": "A105"
}

In [13]:
from numpy.linalg import norm

resident_apartment = {
    "Ravi": "A101",
    "Lakshmi": "A101",
    "Priya": "A102",
    "Suresh": "A102",
    "Arjun": "A103",
    "Lokes": "A104",
    "Ragunath": "A105"
}

def verify_visitor(visitor_image):

    visitor_embedding = get_embedding(visitor_image)

    if visitor_embedding is None:
        return "No Face Detected"

    best_match = None
    best_score = 999

    for resident_name, resident_embedding in resident_db.items():

        distance = norm(
            visitor_embedding - resident_embedding
        )

        if distance < best_score:
            best_score = distance
            best_match = resident_name

    if best_score < 1.0:

        apartment = resident_apartment[best_match]

        return f"""
AUTHORIZED

Resident : {best_match}
Apartment : {apartment}
Score : {best_score:.2f}
"""

    else:

        return f"""
UNAUTHORIZED

Score : {best_score:.2f}
"""

| Feature                | Status |
| ---------------------- | ------ |
| FaceNet Model          | ✅      |
| Resident Database      | ✅      |
| 7 Residents Registered | ✅      |
| Visitor Verification   | ✅      |
| Apartment Mapping      | ✅      |
| Authorized Detection   | ✅      |


In [22]:
from numpy.linalg import norm

resident_apartment = {
    "Ravi": "A101",
    "Lakshmi": "A101",
    "Priya": "A102",
    "Suresh": "A102",
    "Arjun": "A103",
    "Lokes": "A104",
    "Ragunath": "A105"
}

def verify_visitor(visitor_image):

    visitor_embedding = get_embedding(visitor_image)

    if visitor_embedding is None:
        return "No Face Detected"

    best_match = None
    best_score = 999

    for resident_name, resident_embedding in resident_db.items():

        distance = norm(
            visitor_embedding - resident_embedding
        )

        if distance < best_score:
            best_score = distance
            best_match = resident_name

    # Strict threshold
    THRESHOLD = 0.75

    if best_score < THRESHOLD:

        apartment = resident_apartment[best_match]

        return f"""
AUTHORIZED, Allowed


Resident  : {best_match}
Apartment : {apartment}
Score     : {best_score:.4f}
"""

    else:

        return f"""
UNAUTHORIZED, Not Allowed

Unknown Visitor
Closest Match : {best_match}
Score         : {best_score:.4f}
"""

Install Gradio

In [16]:
!pip install gradio

In [17]:
def securegate_app(image):

    image.save("temp_visitor.jpg")

    result = verify_visitor("temp_visitor.jpg")

    return result

In [18]:
def securegate_app(image):

    image.save("temp_visitor.jpg")

    result = verify_visitor("temp_visitor.jpg")

    if "UNAUTHORIZED" in result:

        return f"""
        <div style="
        background-color:#DC2626;
        color:white;
        padding:20px;
        border-radius:12px;
        font-size:20px;
        font-weight:bold;
        line-height:1.8;">

        🚫 ACCESS DENIED

        <br><br>

        {result}

        </div>
        """

    else:

        return f"""
        <div style="
        background-color:#16A34A;
        color:white;
        padding:20px;
        border-radius:12px;
        font-size:20px;
        font-weight:bold;
        line-height:1.8;">

        ✅ ACCESS GRANTED

        <br><br>

        {result}

        </div>
        """

In [19]:
import gradio as gr

app = gr.Interface(
    fn=securegate_app,

    inputs=gr.Image(
        type="pil",
        label="📸 Upload Visitor Photo"
    ),

    outputs=gr.HTML(
        label="🔍 Verification Result"
    ),

    title="""
    <h1 style='
    text-align:center;
    font-size:40px;
    font-weight:bold;
    color:#1E3A8A;'>
    🔐 SecureGate AI
    </h1>
    """,

    description="""
    <div style='
    font-size:20px;
    font-weight:bold;
    text-align:center;
    color:#1F2937;'>

    🔐 Face Recognition |
    👤 Resident Identification |
    🏠 Apartment Mapping |
    🛡️ Visitor Verification

    </div>
    """,

    theme=gr.themes.Soft()
)

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c041fee2e873557f1b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
